# A* vs MCTS (Design-based)
This notebook compares A* and MCTS **on Design.py only**.

## Imports

In [1]:

try:
    import os
    import glob
    import time
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    import graphical_sampling as gs
    from package_sampling.utils import inclusion_probabilities
    from graphical_sampling.design import Design
    from graphical_sampling.criteria.var_nht import VarNHT
    from graphical_sampling.search.astar import AStar
    from graphical_sampling.search.mcts import MCTS
    from graphical_sampling.validation.design_validity import (
        check_probability_sum,
        check_fip_consistency,
    )
    print(AStar)
    print(MCTS)
    print(Design)
    print(VarNHT)
    print(check_probability_sum)
    print(check_fip_consistency)
except KeyError as e:
    print(e)

<class 'graphical_sampling.search.astar.AStar'>
<class 'graphical_sampling.search.mcts.MCTS'>
<class 'graphical_sampling.design.Design'>
<class 'graphical_sampling.criteria.var_nht.VarNHT'>
<function check_probability_sum at 0x000001C999A95750>
<function check_fip_consistency at 0x000001C999A955A0>


## Load population MU284_filtered.csv


In [2]:
try:
    df = pd.read_csv('MU284_filtered.csv')
    # استخراج ستون‌ها از دیتافریم اصلی (df)
    cs82 = df['CS82'].values.astype(float)  # متغیر سایز
    ss82 = df['SS82'].values.astype(float)  # متغیر هدف

    N = len(df)
    n = 20  # طبق مقاله (5, 10, 15)
    rng = np.random.default_rng(42)

    print(f"✅ Data loaded. Population Size (N): {N}, Sample Size (n): {n}")

except FileNotFoundError:
    print("❌ Error: File 'MU284_filtered.csv' not found. Please upload it.")
except KeyError as e:
    print(f"❌ Error: Column {e} not found in the CSV.")

✅ Data loaded. Population Size (N): 281, Sample Size (n): 20


## Select dataset

In [3]:
# محاسبه احتمالات متناسب با سایز (PPS)
probs = n * (cs82 / cs82.sum())

# اگر احتمالی بیشتر از ۱ شد، آن را ۱ می‌کنیم
probs = np.minimum(probs, 1.0)

# نرمال‌سازی مجدد تا مجموع دقیقاً برابر n شود
probs = probs * (n / probs.sum())
#print(probs)
print("✅ Inclusion probabilities constructed.")
print(f"   Sum of probs: {probs.sum():.5f} (Target: {n})")

✅ Inclusion probabilities constructed.
   Sum of probs: 20.00000 (Target: 20)


## Build initial Design (Design.py)

In [4]:
# 1. ساخت طرح گرافیکی اولیه
initial_design = Design(probs, rng=rng)

# 2. تعریف تابع هزینه (واریانس هورویتز-تامپسون روی متغیر SS82)
criteria = VarNHT(ss82,probs)

# 3. محاسبه واریانس اولیه
initial_var = criteria(initial_design)

print(f"🔵 Initial Variance (Random Design): {initial_var:.6f}")

🔵 Initial Variance (Random Design): 241611.839002


## Run A* (Design-based)

In [13]:
try:
    astar = AStar(
        initial_design=initial_design,
        criteria=criteria,
        switch_coefficient=0.5,
        random_pull= False,
        threshold=1e-6,
    )
    t0 = time.perf_counter()
    astar_iters = astar.run(
        max_iterations=500,
        num_new_nodes=10,
        max_open_set_size=1000,
        num_changes=1,
    )
    elapsed_astar = time.perf_counter() - t0

    print("A* iterations:", astar_iters)
    print("A* best VarNHT:", astar.best_criteria_value)
    print("A* time (s):", round(elapsed_astar, 3))
except KeyError as e:
    print(e)

A* iterations: 500
A* best VarNHT: 233983.1524290517
A* time (s): 0.495


## Run MCTS (Design-based)

In [9]:
import time
# =====================================================
# 1) Run MCTS
# =====================================================
t0 = time.perf_counter()

mcts = MCTS(
        initial_design,
        criteria,
        base_changes= 10
)

best_design, best_value = mcts.run(
        max_iterations=100,
        max_children_per_node=20,
        rollout_depth=50
)


elapsed_mcts = time.perf_counter() - t0

print("MCTS best VarNHT:", best_value)
print("MCTS time (s):", round(elapsed_mcts, 3))



=== MCTS INITIALIZED ===
Initial Var: 241611.83900216222
------------------------
=== MCTS RUN STARTED ===
Max iterations: 100
------------------------

>>> ITERATION 0
  SELECTED NODE | depth = 0 | visits = 0 | children = 0
  EXPAND | depth = 0 | rollout_improved = False | mutations = 11
length of node.children:  1
  SIMULATE | start Var = 246732.05828019977
  ROLLOUT RESULT | best Var = 245679.40274627507 | reward = -0.008347294996557146
  BACKPROP | reward = -0.008347294996557146 | from depth = 1
  EXPAND | depth = 0 | rollout_improved = True | mutations = 9
length of node.children:  2
  SIMULATE | start Var = 251565.47878667712
  ROLLOUT RESULT | best Var = 249286.33776032925 | reward = -0.015633585785102922
  BACKPROP | reward = -0.015633585785102922 | from depth = 1
  EXPAND | depth = 0 | rollout_improved = True | mutations = 9
length of node.children:  3
  SIMULATE | start Var = 250229.25224047154
  ROLLOUT RESULT | best Var = 250229.25224047154 | reward = -0.01752072649427781
 

## Final comparison

In [15]:
print("========== FINAL COMPARISON ==========")
print("Initial VarNHT :", initial_var)
print("A* VarNHT      :", astar.best_criteria_value)
print("MCTS VarNHT    :", best_value)


========== FINAL COMPARISON ==========
Initial VarNHT : 241611.83900216222
A* VarNHT      : 233983.1524290517
MCTS VarNHT    : 230712.92742646486
